# Chapter 23: Estimating Volatilities and Correlations

> **Color Convention**  
> 🟡 **Yellow** = Definition / Theorem / Property  
> 🟢 **Green** = Comment / Application / Insight

---

## Overview

Volatilities and correlations cluster and change over time. This chapter covers: standard historical estimation, EWMA, and GARCH(1,1) — models that recognize time-varying volatility.

---

## 23.1 Estimating Volatility

🟡 **Definition:**

> Let $\sigma_n$ = volatility on day $n$ estimated at end of day $n-1$. The **variance rate** = $\sigma_n^2$.
>
> Define $u_i = \ln(S_i / S_{i-1})$ (log return on day $i$).
>
> **Unbiased historical variance estimate** using the most recent $m$ observations:
>
> $$\sigma_n^2 = \frac{1}{m-1}\sum_{i=1}^{m}(u_{n-i} - \bar{u})^2$$

🟡 **Simplified formula** (setting $\bar{u} = 0$, $m-1 \to m$):

$$\sigma_n^2 = \frac{1}{m}\sum_{i=1}^{m} u_{n-i}^2$$

🟡 **General weighted scheme:**

$$\sigma_n^2 = \sum_{i=1}^{m} \alpha_i u_{n-i}^2, \quad \sum_i \alpha_i = 1, \quad \alpha_i > 0$$

More weight on recent observations → better tracking of time-varying volatility.

---

## 23.2 The EWMA Model

🟡 **Definition:**

> In the **Exponentially Weighted Moving Average (EWMA)** model, weights decay geometrically: $\alpha_{i+1} = \lambda \alpha_i$ where $0 < \lambda < 1$.
>
> This yields the elegant **updating formula**:
>
> $$\sigma_n^2 = \lambda \sigma_{n-1}^2 + (1-\lambda) u_{n-1}^2$$

🟢 **Comment:**

> EWMA requires storing only the current variance estimate and the most recent return — very memory efficient. When a large return $u_{n-1}$ occurs, $\sigma_n^2$ jumps immediately and then decays back at rate $\lambda$ per day.
>
> RiskMetrics uses $\lambda = 0.94$ for daily data. The effective number of days used in the estimate is $\approx 1/(1-\lambda)$.

🟢 **Comment:**

> EWMA is designed to **track changes in volatility quickly**. The impact of a shock $u_{n-1}^2$ decays exponentially: its weight at lag $k$ is $(1-\lambda)\lambda^{k-1}$, so shocks become negligible after $\approx 1/(1-\lambda)$ days.

---

## 23.3 The GARCH(1,1) Model

🟡 **Definition:**

> **GARCH(1,1)** (Bollerslev, 1986) adds mean-reversion to a long-run variance $V_L$:
>
> $$\sigma_n^2 = \omega + \alpha u_{n-1}^2 + \beta \sigma_{n-1}^2$$
>
> where $\omega = \gamma V_L$ and $\gamma + \alpha + \beta = 1$. Equivalently:
>
> $$\sigma_n^2 = \gamma V_L + \alpha u_{n-1}^2 + \beta \sigma_{n-1}^2$$

🟢 **Comment:**

> **GARCH(1,1) exhibits mean reversion**: the variance is pulled back toward the long-run level $V_L$. The speed of reversion is $1 - \alpha - \beta$. EWMA is a special case with $\gamma = 0$ (no long-run mean).
>
> The variance process is equivalent to:
> $$dV = a(V_L - V)\,dt + \xi V\,dz$$
> an Ornstein–Uhlenbeck (mean-reverting) process — the same structure as the CIR interest rate model.

---

## 23.4 Choosing Between Models

GARCH(1,1) is preferred when volatility shows strong mean reversion. EWMA suffices when mean reversion is weak or the horizon is short. Both outperform equally-weighted historical estimates for VaR.

---

## 23.5 Maximum Likelihood Methods

🟡 **Property (Log-Likelihood):**

> Estimate parameters by maximizing the log-likelihood of observed returns:
>
> $$\ln L = -\sum_{i=1}^{n}\left[\ln(\sigma_i^2) + \frac{u_i^2}{\sigma_i^2}\right]$$
>
> Optimization is done numerically. For GARCH(1,1), optimize over $(\omega, \alpha, \beta)$.

🟡 **Variance targeting:**

> A robust alternative: set $V_L = $ sample variance (match the long-run mean to data), then estimate $\alpha$ and $\beta$ via MLE. More stable when data is limited.

🟡 **Property:**

> Per-day volatilities are typically around 1–2%, but can reach 8%/day in crises (e.g., March 2020).

---

## 23.6 Using GARCH(1,1) to Forecast Future Volatility

🟡 **Property (Multi-step Forecast):**

> Expected variance on day $n + t$ given today's estimate:
>
> $$E[\sigma_{n+t}^2] = V_L + (\alpha + \beta)^t (\sigma_n^2 - V_L)$$
>
> As $t \to \infty$: $E[\sigma_{n+t}^2] \to V_L$ (mean reversion). The term structure of volatility is downward-sloping when $\sigma_n^2 > V_L$ and upward-sloping when $\sigma_n^2 < V_L$.

🟡 **Property (Ljung–Box Test):**

> To test model adequacy, the **Ljung–Box statistic** tests whether standardized residuals $u_i^2/\sigma_i^2$ are uncorrelated:
>
> $$Q = m\sum_{k=1}^{K} w_k \hat{h}_k^2, \quad w_k = \frac{m+2}{m-k}$$
>
> For $K = 15$: reject zero autocorrelation at 95% confidence if $Q > 25$.

---

## 23.7 Correlations

🟡 **Definition (EWMA for Covariance):**

> Define $x_i = (X_i - X_{i-1})/X_{i-1}$ and $y_i = (Y_i - Y_{i-1})/Y_{i-1}$. EWMA covariance:
>
> $$\text{cov}_n = \lambda\,\text{cov}_{n-1} + (1-\lambda) x_{n-1} y_{n-1}$$

🟡 **Definition (Correlation Estimate):**

> $$\hat{\rho}_n = \frac{\text{cov}_n}{\sigma_{x,n}\,\sigma_{y,n}}$$

🟡 **Property (Positive Semi-Definite Consistency):**

> An $N \times N$ variance-covariance matrix $\Omega$ is internally consistent (positive semi-definite) if and only if:
>
> $$\mathbf{w}^T \Omega\, \mathbf{w} \geq 0 \quad \forall\, \mathbf{w} \in \mathbb{R}^N$$
>
> If some correlations are estimated over different periods, the resulting matrix may violate this condition → must be "repaired" (e.g., by finding the nearest PSD matrix).

---

## Summary

| Model | Updating Formula | Key Feature |
|-------|-----------------|-------------|
| Historical | $\sigma_n^2 = \frac{1}{m}\sum u_i^2$ | Equal weight; simple |
| EWMA | $\sigma_n^2 = \lambda\sigma_{n-1}^2 + (1-\lambda)u_{n-1}^2$ | Exponential decay; efficient |
| GARCH(1,1) | $\sigma_n^2 = \omega + \alpha u_{n-1}^2 + \beta\sigma_{n-1}^2$ | Mean reversion to $V_L$ |
